# Notebook 2 — Baseline Food Classifier (Without GAN)

**Project:** An Improved Computer Vision Model for Food Classification in Smart Refrigerators using GAN-Based Data Augmentation  
**Author:** Premshakthi Sekar | MSc Artificial Intelligence — Northumbria University  

---

## Purpose
This notebook builds and evaluates the **baseline CNN classifier** using the original dataset — before any GAN-based augmentation. This establishes the performance benchmark that the GAN-augmented model (Notebook 4) is compared against.

## Model Architecture
- **Base model:** MobileNetV2 (pre-trained on ImageNet) — used for transfer learning
- **Classification head:** GlobalAveragePooling2D + Dense(30, softmax)
- **Classes:** 30 food item categories
- **Input size:** 640×640×3

## Why MobileNetV2?
MobileNetV2 is a lightweight, efficient architecture well-suited for image classification tasks with limited data. Using pre-trained ImageNet weights through transfer learning allows the model to leverage existing visual feature representations rather than learning from scratch.

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print('All libraries imported successfully.')

## Step 2: Load & Preprocess Dataset

Images are loaded from class subdirectories (organised in Notebook 1), resized to 640×640, and converted to numpy arrays.

In [ ]:
# Path to dataset — update to match your local path
dataset_path = 'dataset/images'

classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
print(f'Found {len(classes)} classes: {sorted(classes)}')

images = []
labels = []

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)
    for filename in os.listdir(class_dir):
        filepath = os.path.join(class_dir, filename)
        if filepath.endswith('.jpg') or filepath.endswith('.png'):
            img = Image.open(filepath).convert('RGB')
            img = img.resize((640, 640))
            images.append(np.array(img))
            labels.append(class_name)

print(f'\nTotal images loaded: {len(images)}')
print(f'Total labels: {len(labels)}')

## Step 3: Encode Labels & Split Dataset

In [ ]:
# Encode string class labels to integers
label_encoder = LabelEncoder()
numerical_labels = label_encoder.fit_transform(labels)

print(f'Label encoding complete. Classes: {list(label_encoder.classes_)}')

# Train/test split — 80% train, 20% test, stratified to maintain class balance
x_train, x_test, y_train, y_test = train_test_split(
    np.array(images), numerical_labels,
    test_size=0.2,
    stratify=numerical_labels,
    random_state=42
)

# Normalise pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

print(f'\nDataset split:')
print(f'  x_train: {x_train.shape}')
print(f'  x_test:  {x_test.shape}')

## Step 4: Build the Model — Transfer Learning with MobileNetV2

The base MobileNetV2 layers are frozen (not retrained). Only the classification head is trained on our food dataset.

In [ ]:
# Load MobileNetV2 with ImageNet weights, exclude top classification layer
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(640, 640, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base model weights

# Build full model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(30, activation='softmax')  # 30 food classes
])

# Compile
model.compile(
    optimizer=Adam(),
    loss=SparseCategoricalCrossentropy(),
    metrics=[SparseCategoricalAccuracy()]
)

model.summary()

## Step 5: Train the Baseline Model

The model is trained for 200 epochs with a 10% validation split. This is the baseline — trained on the original dataset without any GAN augmentation.

In [ ]:
# Train the model
# Note: This takes significant time depending on hardware
history = model.fit(
    x_train, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.1
)

print('Training complete.')

## Step 6: Evaluate the Baseline Model

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Baseline Model — Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')
print(f'Baseline Model — Test Loss:     {test_loss:.4f}')

In [ ]:
# Plot training and validation accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['sparse_categorical_accuracy'], label='Train', color='#2196F3')
ax1.plot(history.history['val_sparse_categorical_accuracy'], label='Validation', color='#FF5722')
ax1.set_title('Baseline Model — Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'], label='Train', color='#2196F3')
ax2.plot(history.history['val_loss'], label='Validation', color='#FF5722')
ax2.set_title('Baseline Model — Loss', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

In [ ]:
# Classification report per class
y_pred_probs   = model.predict(x_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

print('Baseline Classification Report:')
print('=' * 60)
print(classification_report(
    y_test, y_pred_classes,
    target_names=label_encoder.classes_
))

---
## Summary

| Metric | Baseline (No GAN) |
|--------|------------------|
| Test Accuracy | ~51% (see results above) |
| Key Issue | Significant overfitting observed — high train accuracy, low validation accuracy |
| Root Cause | Limited and imbalanced training data |

**Observation:** The baseline model showed commendable training accuracy but suffered from overfitting and poor generalisation to unseen data. This motivates the use of GAN-based data augmentation.

**Next:** Notebook 3 — GAN Training for Synthetic Data Generation